# Developing python implementation of BASL

In [7]:
## Setting "base directory" for allowing imports
import os
import sys
import warnings

base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditData, CreditDataGenerator
from berebasl.BASL.BASL import accept_based_on_top_percentent_of_arbitrary_var
from berebasl.BASL.classifiers import TorchLogistic

import torch

In [2]:
torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

initial_seed = 1807

#Acceptance loop parametrization
init_sample = 100
sample_size = 100
holdout_sample = 3000
num_gens = 300
top_percent = 0.2

determinstic_mixture_weights = True

data_gen = CreditDataGenerator.init_with_internal_logic(
    count_covariates=2,
    mean_bad_diff=torch.tensor([1.0,2.0]),
    covars = {
        "bad" : torch.tensor([[1.0, 0.2], [0.2,1.0]]),
        "good" : torch.tensor([[1.0,-0.2], [-0.2,1.0]])
    },
    iid = False,
    mixture_weights=None,
    bad_ratio = 0.5,
    noise_var=0.0,
    device = device,
    seed_credit_data_gen=initial_seed,
    #dtype=torch.get_default_dtype()
)

# Initial population
data_gen.manual_seed(initial_seed)
features, default_flag = data_gen.sample(100)

accepts = accept_based_on_top_percentent_of_arbitrary_var(
    features, 
    default_flag,
    var_for_rule=0,
    top_percent=top_percent,
    default_value = CreditDataGenerator.bad_good_encoding["bad"]
)

credit_data = CreditData(features, default_flag, accepts)

## Understanding `biasAwareSelfLearning`

The signature:

```r
biasAwareSelfLearning <- function(accepts,
                                  rejects,
                                  target           = 'BAD', 
                                  filtering_beta   = c(0, 1),
                                  weak_learner     = 'classif.logreg',
                                  strong_learner   = 'classif.logreg',
                                  holdout_percent  = 0.1,
                                  sampling_percent = 1,
                                  labeling_percent = 0.01, 
                                  multiplier       = 1, 
                                  max_iterations   = 3,
                                  early_stop       = F,
                                  silent           = F) 
```

The call
```r
# BASL parameters
  filtering_beta   <- c(0.01, 0.99)
  holdout_percent  <- 0.1
  labeling_percent <- 0.1
  sampling_percent <- 0.8
  multiplier       <- 2
  max_iterations   <- 5
  early_stop       <- T
  
  # label rejected cases
  rej_labels <- biasAwareSelfLearning(accepts          = current_accepts, 
                                      rejects          = current_rejects,
                                      target           = 'BAD', 
                                      filtering_beta   = filtering_beta,
                                      weak_learner     = 'classif.logreg',
                                      strong_learner   = 'classif.logreg',
                                      holdout_percent  = holdout_percent,
                                      labeling_percent = labeling_percent, 
                                      sampling_percent = sampling_percent,
                                      multiplier       = multiplier, 
                                      max_iterations   = max_iterations,
                                      early_stop       = early_stop,
                                      silent           = T)
```

There are a couple of parts in the algorithm:

0. **Preparation:** defines several variables. 
    * specially interesting is that it creates "holdout samples" from the accepts when `early_stop == T`
    * it also defines as `bm_metric` the auc.
1. **Reject Filtering** It occurs if `filtering_beta != c(0,1)` (as the call is done inside the acceptance loop). The filtering is done via the function `filteringStage`

In [ ]:
from sklearn.ensemble import IsolationForest
from typing import Callable, Union
import numpy as np

class BASL:
    filtering_beta : torch.Tensor #filtering_beta in algorithm are both of these
    weak_learner : Callable[[torch.Tensor], torch.Tensor]
    strong_learner : Callable[[torch.Tensor], torch.Tensor]
    hold_out_percent : float
    labeling_percent : float
    multiplier : float
    max_iterations :int
    early_stop : bool
    isolation_forest : IsolationForest

    def __init__(
            self,
            filtering_quantiles : dict[str, float], #filtering_beta in algorithm are both of these
            weak_learner : Callable[[torch.Tensor], torch.Tensor],
            strong_learner : Callable[[torch.Tensor], torch.Tensor],
            hold_out_percent : float,
            labeling_percent : float,
            multiplier : float,
            max_iterations :int,
            early_stop : bool,
            isolation_forest : IsolationForest
    ):
        BASL.check_filtering_quantiles(filtering_quantiles)
        self.filtering_quantiles = filtering_quantiles
        
        self.weak_learner = weak_learner
        self.strong_learner = strong_learner
        self.hold_out_percent = float(hold_out_percent)
        if not (0 <= self.hold_out_percent <= 1):
            raise ValueError("hold_out_percent needs to be between 0 and 1")
        self.labeling_percent = float(labeling_percent)
        if not (0 <= self.labeling_percent <= 1):
            raise ValueError("labeling_percent needs to be between 0 and 1")
        
        self.multiplier = float(multiplier)
        self.max_iterations = int(max_iterations)
        if self.max_iterations < 1:
            raise ValueError("max_iterations needs to be at least 1")
        
        self.early_stop = bool(early_stop)
        self.isolation_forest = isolation_forest

    @staticmethod
    def check_filtering_quantiles(filtering_quantiles) -> None:
        if not 0.0 <= filtering_quantiles['lower'] <= 1.0:
            raise ValueError("filtering_quantiles['lower'] must be in the interval [0, 1].")
        if not 0.0 <= filtering_quantiles['lower'] <= 1.0:
            raise ValueError("filtering_quantiles['upper'] must be in the interval [0, 1].")
        if filtering_quantiles['lower'] >= filtering_quantiles['upper']:
            raise ValueError(
                "filtering_quantiles['lower'] must be strictly smaller than filtering_quantiles['upper']."
            )

    @property
    def should_filter(self):
        return (self.filtering_quantiles['lower'] > 0) or (self.filtering_quantiles['upper'] <1)
    
    

    def self_learn(
            self,
            features_accept : torch.Tensor,
            default_flags_accept : torch.Tensor,
            features_reject : torch.Tensor,
            silent : bool = True
    ) -> torch.Tensor:
        pass

    def filter_rejects(
        self,
        features_rejects: torch.Tensor,
        return_index: bool = True,
    ) -> Union[np.ndarray, np.ndarray]:
        """
        Filters observations using a two-sided trimming strategy based on
        Isolation Forest normality scores.

        The function fits an Isolation Forest model on the provided feature
        matrix and computes the negative anomaly scores via
        ``IsolationForest.score_samples``. These scores induce a relative
        normality (similarity) ranking.

        Observations are retained if their score lies within the central
        quantile interval defined by ``lower_trim_quantile`` and
        ``upper_trim_quantile``. Consequently, both highly anomalous
        observations (lower tail) and overly typical observations
        (upper tail) are removed.

        This procedure corresponds to a two-sided percentile-based filtering
        scheme as described in Kozodoi et al. (2025), "Fighting Sampling Bias".

        Args:
            ifo (IsolationForest):
                An unfit ``IsolationForest`` instance used to compute
                normality scores.
            lower_trim_quantile (float):
                Lower quantile boundary in the interval ``[0, 1]``.
                Observations with scores below this quantile are discarded.
            upper_trim_quantile (float):
                Upper quantile boundary in the interval ``[0, 1]``.
                Observations with scores above this quantile are discarded.
            features (np.ndarray):
                Feature matrix of shape ``(n_samples, n_features)``.
            return_index (bool, optional):
                If ``True``, return a boolean mask indicating retained
                observations. If ``False``, return the filtered feature
                matrix. Defaults to ``True``.

        Returns:
            torch.Tensor:
                If ``return_index`` is ``True``, a boolean array of shape
                ``(n_samples,)`` indicating which observations are retained.
                Otherwise, a feature matrix containing only the retained
                observations.
        """
        
        features = features_rejects.detach().numpy()
        self.ifo.fit(features)
        normality_scores = self.ifo.score_samples(features)

        lower_score_bound, upper_score_bound = np.quantile(
            normality_scores,
            [self.filtering_quantiles["lower"], self.filtering_quantiles["upper"]],
        )

        keep_mask = (
                (lower_score_bound <= normality_scores)
                & (normality_scores <= upper_score_bound)
        )
        keep_mask = torch.from_numpy(keep_mask)

        return keep_mask if return_index else features_rejects[keep_mask]

    def bayesian_evaluation( #Next step to implement
            self,
            features_accept : torch.Tensor,
            default_flag_accept : torch.Tensor,
            features_reject : torch.Tensor,
            *args,
            **kwargs
    ) -> torch.Tensor:
        pass
    
    def should_stop_early(self) -> bool:
        pass
    
    def label_rejects(self) -> torch.Tensor:
        pass
    
    
    


### Sensible `__init__` stuff or properties

In [ ]:
import numpy as np

class DummyBASL:
    filtering_beta : torch.Tensor #filtering_beta in algorithm are both of these
    weak_learner : Callable[[torch.Tensor], torch.Tensor]
    strong_learner : Callable[[torch.Tensor], torch.Tensor]
    hold_out_percent : float
    labeling_percent : float
    multiplier : float
    max_iterations :int
    early_stop : bool
    isolation_forest : IsolationForest

    def __init__(
            self,
            filtering_quantiles : dict[str, float], #filtering_beta in algorithm are both of these
            weak_learner : Callable[[torch.Tensor], torch.Tensor],
            strong_learner : Callable[[torch.Tensor], torch.Tensor],
            hold_out_percent : float,
            labeling_percent : float,
            multiplier : float,
            max_iterations :int,
            early_stop : bool,
            isolation_forest : IsolationForest
    ):
        DummyBASL.check_filtering_quantiles(filtering_quantiles)
        self.filtering_quantiles = filtering_quantiles
        
        self.weak_learner = weak_learner
        self.strong_learner = strong_learner
        self.hold_out_percent = float(hold_out_percent)
        if not (0 <= self.hold_out_percent <= 1):
            raise ValueError("hold_out_percent needs to be between 0 and 1")
        self.labeling_percent = float(labeling_percent)
        if not (0 <= self.labeling_percent <= 1):
            raise ValueError("labeling_percent needs to be between 0 and 1")
        
        self.multiplier = float(multiplier)
        self.max_iterations = int(max_iterations)
        if self.max_iterations < 1:
            raise ValueError("max_iterations needs to be at least 1")
        
        self.early_stop = bool(early_stop)
        self.isolation_forest = isolation_forest


    @staticmethod
    def check_filtering_quantiles(filtering_quantiles) -> None:
        if not 0.0 <= filtering_quantiles['lower'] <= 1.0:
            raise ValueError("filtering_quantiles['lower'] must be in the interval [0, 1].")
        if not 0.0 <= filtering_quantiles['lower'] <= 1.0:
            raise ValueError("filtering_quantiles['upper'] must be in the interval [0, 1].")
        if filtering_quantiles['lower'] >= filtering_quantiles['upper']:
            raise ValueError(
                "filtering_quantiles['lower'] must be strictly smaller than filtering_quantiles['upper']."
            )

    @property
    def should_filter(self):
        return (self.filtering_quantiles['lower'] > 0) or (self.filtering_quantiles['upper'] <1)
    
    def filter_rejects(
        self,
        lower_trim_quantile: float,
        upper_trim_quantile: float,
        features_rejects: torch.Tensor,
        return_index: bool = True,
    ) -> Union[np.ndarray, np.ndarray]:
        """
        Filters observations using a two-sided trimming strategy based on
        Isolation Forest normality scores.

        The function fits an Isolation Forest model on the provided feature
        matrix and computes the negative anomaly scores via
        ``IsolationForest.score_samples``. These scores induce a relative
        normality (similarity) ranking.

        Observations are retained if their score lies within the central
        quantile interval defined by ``lower_trim_quantile`` and
        ``upper_trim_quantile``. Consequently, both highly anomalous
        observations (lower tail) and overly typical observations
        (upper tail) are removed.

        This procedure corresponds to a two-sided percentile-based filtering
        scheme as described in Kozodoi et al. (2025), "Fighting Sampling Bias".

        Args:
            ifo (IsolationForest):
                An unfit ``IsolationForest`` instance used to compute
                normality scores.
            lower_trim_quantile (float):
                Lower quantile boundary in the interval ``[0, 1]``.
                Observations with scores below this quantile are discarded.
            upper_trim_quantile (float):
                Upper quantile boundary in the interval ``[0, 1]``.
                Observations with scores above this quantile are discarded.
            features (np.ndarray):
                Feature matrix of shape ``(n_samples, n_features)``.
            return_index (bool, optional):
                If ``True``, return a boolean mask indicating retained
                observations. If ``False``, return the filtered feature
                matrix. Defaults to ``True``.

        Returns:
            np.ndarray:
                If ``return_index`` is ``True``, a boolean array of shape
                ``(n_samples,)`` indicating which observations are retained.
                Otherwise, a feature matrix containing only the retained
                observations.

        Raises:
            ValueError:
                If ``lower_trim_quantile`` or ``upper_trim_quantile`` are
                outside the interval ``[0, 1]`` or if
                ``lower_trim_quantile >= upper_trim_quantile``.
        """
        if not 0.0 <= lower_trim_quantile <= 1.0:
            raise ValueError("lower_trim_quantile must be in the interval [0, 1].")
        if not 0.0 <= upper_trim_quantile <= 1.0:
            raise ValueError("upper_trim_quantile must be in the interval [0, 1].")
        if lower_trim_quantile >= upper_trim_quantile:
            raise ValueError(
                "lower_trim_quantile must be strictly smaller than upper_trim_quantile."
            )

        self.ifo.fit(features)
        normality_scores = self.ifo.score_samples(features)

        lower_score_bound, upper_score_bound = np.quantile(
            normality_scores,
            [lower_trim_quantile, upper_trim_quantile],
        )

        keep_mask = (
                (lower_score_bound <= normality_scores)
                & (normality_scores <= upper_score_bound)
        )

        return keep_mask if return_index else features[keep_mask]

### Understanding `filteringStage`

Signature:

```r
filteringStage <- function(accepts, 
                           rejects, 
                           target    = 'BAD', 
                           beta      = c(0, 1),
                           num_trees = 100)
```

Call:

```r
filter_idx <- filteringStage(accepts   = train, # ifelse(early_stop, accepts[-holdout_idx_accepts, ], accepts)
                             rejects   = test,  # ifelse(early_stop, rejects[-holdout_idx_rejects, ], rejects)
                             beta      = filtering_beta, # From loop: c(0.01,0.99)
                             num_trees = 100)
```

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

from typing import Union

forest = IsolationForest(n_estimators=100, max_samples="auto", random_state=1807)
def filter(
    self,
    lower_trim_quantile: float,
    upper_trim_quantile: float,
    features: np.ndarray,
    return_index: bool = True,
) -> Union[np.ndarray, np.ndarray]:
    """
    Filters observations using a two-sided trimming strategy based on
    Isolation Forest normality scores.

    The function fits an Isolation Forest model on the provided feature
    matrix and computes the negative anomaly scores via
    ``IsolationForest.score_samples``. These scores induce a relative
    normality (similarity) ranking.

    Observations are retained if their score lies within the central
    quantile interval defined by ``lower_trim_quantile`` and
    ``upper_trim_quantile``. Consequently, both highly anomalous
    observations (lower tail) and overly typical observations
    (upper tail) are removed.

    This procedure corresponds to a two-sided percentile-based filtering
    scheme as described in Kozodoi et al. (2025), "Fighting Sampling Bias".

    Args:
        ifo (IsolationForest):
            An unfit ``IsolationForest`` instance used to compute
            normality scores.
        lower_trim_quantile (float):
            Lower quantile boundary in the interval ``[0, 1]``.
            Observations with scores below this quantile are discarded.
        upper_trim_quantile (float):
            Upper quantile boundary in the interval ``[0, 1]``.
            Observations with scores above this quantile are discarded.
        features (np.ndarray):
            Feature matrix of shape ``(n_samples, n_features)``.
        return_index (bool, optional):
            If ``True``, return a boolean mask indicating retained
            observations. If ``False``, return the filtered feature
            matrix. Defaults to ``True``.

    Returns:
        np.ndarray:
            If ``return_index`` is ``True``, a boolean array of shape
            ``(n_samples,)`` indicating which observations are retained.
            Otherwise, a feature matrix containing only the retained
            observations.

    Raises:
        ValueError:
            If ``lower_trim_quantile`` or ``upper_trim_quantile`` are
            outside the interval ``[0, 1]`` or if
            ``lower_trim_quantile >= upper_trim_quantile``.
    """
    if not 0.0 <= lower_trim_quantile <= 1.0:
        raise ValueError("lower_trim_quantile must be in the interval [0, 1].")
    if not 0.0 <= upper_trim_quantile <= 1.0:
        raise ValueError("upper_trim_quantile must be in the interval [0, 1].")
    if lower_trim_quantile >= upper_trim_quantile:
        raise ValueError(
            "lower_trim_quantile must be strictly smaller than upper_trim_quantile."
        )

    self.ifo.fit(features)
    normality_scores = self.ifo.score_samples(features)

    lower_score_bound, upper_score_bound = np.quantile(
        normality_scores,
        [lower_trim_quantile, upper_trim_quantile],
    )

    keep_mask = (
        (lower_score_bound <= normality_scores)
        & (normality_scores <= upper_score_bound)
    )

    return keep_mask if return_index else features[keep_mask]
keep_mask = filter(forest, lower_trim_quantile=0.01, upper_trim_quantile=0.99, features=features.numpy())


### Developing the flow of the original algorithm

In [ ]:
data = credit_data.to_sample_dataset()

self = DummyBASL(
    filtering_beta={"lower" : .01, "upper" : .99},
    weak_learner = TorchLogistic(
        n_features=credit_data.features.shape[-1],
        n_classes=2
    ),
    strong_learner = TorchLogistic(
        n_features=credit_data.features.shape[-1],
        n_classes=2
    ),
    hold_out_percent = 0.1,
    labeling_percent=0.01,
    multiplier=1,
    max_iterations=3,
    early_stop=True,
    isolation_forest=IsolationForest(n_estimators=100, max_samples="auto", random_state=1807)
)

#### Flow of the algorithm

In [ ]:


if True:
    if True:
        if self.early_stop:
            data, hold_out_data = data.train_test_split(self.hold_out_percent)

        # percentage
        per_b = self.labeling_percent
        per_g = self.labeling_percent / self.multiplier

        something_that_happens_in_between = """
            train    <- accepts
            test     <- rejects
            raw_test <- rejects
            rm(list = c('accepts', 'rejects'))
        """

        eval_params = """
            bm_metric   <- mlr::auc
            bm_min_iter <- 10^2
            bm_max_iter <- 10^4
            bm_epsilon  <- 10^4
        """

        

